<a href="https://colab.research.google.com/github/Hackathon-D-Group-2026/Hackathon_D_Group_2026/blob/main/hackathon_seasonal_flu_final_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Seasonal Flu Vaccine Acceptance & Town Cost Minimization Pipeline

## Executive Summary & Business Problem
This notebook presents an end-to-end Machine Learning, Data Exploration, Preprocessing, Modeling, Hyperparameter Tuning, Cross-Validation, and Decision Threshold Optimization pipeline to solve the town seasonal flu management challenge.

### Problem Formulation & Cost Metric
The goal is to **minimize total town expenditure**, which comprises two main components:
1. **Vaccine Costs ($300 per vaccine):** Vaccines sent to residents' homes are ruined if not used. Therefore, the town must send vaccines **only to people who will accept them**. If a vaccine is offered/sent to someone who wants it, it costs $300. If an available vaccine is NOT sent to someone who wants it, by local law the town must pay the individual the cost of the vaccine ($300 penalty). If a vaccine is sent to someone who does NOT want it, the vaccine is wasted ($300 loss).
2. **Hospitalization Costs ($18,500 per hospitalization):** Assuming 100% infection probability (due to winter indoor parties), each individual has an estimated risk of hospitalization based on chronic medical conditions and age. Vaccination reduces baseline hospitalization probability significantly (by ~80%).

$$\text{Total Financial Cost} = \sum_{i} \left[ C_{\text{vaccine}}(i) + C_{\text{hosp}}(i) \right]$$


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, f1_score, confusion_matrix, roc_curve
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Set global styles & random state
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.size'] = 11
plt.rcParams['figure.titlesize'] = 14

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

COST_VACCINE = 300.0
COST_HOSPITAL = 18500.0

print("Setup completed successfully.")


## 1. Data Loading & Exploratory Data Analysis (EDA)

We load `dataset_A_training.csv` (4,756 records) and `dataset_A_testing.csv` (4,749 records).
We inspect data types, missing value distributions, class balances, and feature correlations with vaccine uptake (`seasonal_vaccine`).

In [ ]:
# Load Datasets
train_path = 'dataset_A_training.csv'
test_path = 'dataset_A_testing.csv'

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

print(f"Training set shape: {df_train.shape}")
print(f"Testing set shape:  {df_test.shape}")

# Inspect target variable balance
print("\nTarget Variable ('seasonal_vaccine') Distribution:")
print(df_train['seasonal_vaccine'].value_counts(normalize=True))

# Plot Target Class Distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='seasonal_vaccine', data=df_train, palette=['#e74c3c', '#2ecc71'])
plt.title('Target Distribution: Seasonal Vaccine Uptake')
plt.xlabel('Vaccine Accepted (0 = No, 1 = Yes)')
plt.ylabel('Count')
plt.xticks([0, 1], ['No (0)', 'Yes (1)'])
plt.tight_layout()
plt.show()


In [ ]:
# Visualizing Missing Values across Features
missing_train = df_train.isnull().sum()
missing_train = missing_train[missing_train > 0].sort_values(ascending=False)

plt.figure(figsize=(12, 5))
sns.barplot(x=missing_train.values, y=missing_train.index, palette='crest')
plt.title('Missing Value Count per Feature (Training Data)')
plt.xlabel('Number of Missing Values')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


In [ ]:
# Exploratory Plots: Key Drivers of Vaccine Acceptance
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Doctor Recommendation vs Acceptance
sns.barplot(x='doctor_recc_seasonal', y='seasonal_vaccine', data=df_train, ax=axes[0], palette='Blues')
axes[0].set_title('Vaccine Acceptance by Doctor Recommendation')
axes[0].set_xlabel('Doctor Recommended (0=No, 1=Yes)')
axes[0].set_ylabel('Acceptance Rate')

# 2. Opinion on Vaccine Effectiveness vs Acceptance
sns.barplot(x='opinion_seas_vacc_effective', y='seasonal_vaccine', data=df_train, ax=axes[1], palette='Greens')
axes[1].set_title('Vaccine Acceptance by Perceived Effectiveness')
axes[1].set_xlabel('Effectiveness Rating (1=Low, 5=High)')
axes[1].set_ylabel('Acceptance Rate')

# 3. Opinion on Risk of Sickness vs Acceptance
sns.barplot(x='opinion_seas_risk', y='seasonal_vaccine', data=df_train, ax=axes[2], palette='Oranges')
axes[2].set_title('Vaccine Acceptance by Perceived Flu Risk')
axes[2].set_xlabel('Perceived Risk (1=Low, 5=High)')
axes[2].set_ylabel('Acceptance Rate')

plt.tight_layout()
plt.show()


## 2. Preprocessing & Feature Engineering

- **Numerical / Ordinal Features:** Imputed using `SimpleImputer` (median) and scaled.
- **Categorical Features:** Imputed using `SimpleImputer` (most_frequent) and one-hot encoded (`OneHotEncoder`).
- **Data Transformation:** Pipelines constructed to prevent data leakage during Cross-Validation.

In [ ]:
# Categorize features
target_col = 'seasonal_vaccine'
id_col = 'respondent_id'

feature_cols = [c for c in df_train.columns if c not in [id_col, target_col]]

num_cols = df_train[feature_cols].select_dtypes(include=['float64', 'int64']).columns.tolist()
cat_cols = df_train[feature_cols].select_dtypes(include=['object']).columns.tolist()

print(f"Numerical Features ({len(num_cols)}): {num_cols}")
print(f"Categorical Features ({len(cat_cols)}): {cat_cols}")

# Define Preprocessing Transformers
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ]
)

# Fit preprocessor on training data
X_raw = df_train[feature_cols]
y = df_train[target_col].values

X_processed = preprocessor.fit_transform(X_raw)
X_test_processed = preprocessor.transform(df_test[feature_cols])

print(f"Processed Training Feature Matrix Shape: {X_processed.shape}")
print(f"Processed Testing Feature Matrix Shape:  {X_test_processed.shape}")


## 3. Hospitalization Risk Model & Custom Metric Definition

Per the hackathon target metric rules:
- Baseline hospitalization risk depends on age and chronic conditions (`number_chronic_med_conditions`).
- Receiving the vaccine reduces hospitalization risk by 80%.
- We define a custom function `compute_total_town_cost` that calculates total town costs under any decision threshold.

In [ ]:
def estimate_hospitalization_risk_vectorized(df_raw, vaccination_status_vec):
    """
    Estimates individual baseline hospitalization risk based on chronic conditions and age group.
    Vaccination reduces baseline risk by 80%.
    """
    chronic = df_raw['number_chronic_med_conditions'].fillna(0).values
    age = df_raw['age_group'].fillna('35 - 44 Years').values

    # Base risk calculation
    base_risk = 0.04 + (chronic * 0.08)
    age_multiplier = np.where(age == '65+ Years', 1.8, np.where(age == '55 - 64 Years', 1.3, 1.0))
    base_risk = base_risk * age_multiplier

    # Reduced risk if vaccinated
    effective_risk = np.where(vaccination_status_vec == 1, base_risk * 0.20, base_risk)
    return np.clip(effective_risk, 0.005, 0.90)

def compute_town_cost_for_predictions(df_raw, y_true, y_prob_accept, threshold=0.5):
    """
    Computes exact financial cost incurred by the town based on predicted vaccine offer decisions.
    """
    send_vaccine = (y_prob_accept >= threshold).astype(int)
    N = len(y_true)

    # Vaccine cost calculation
    # 1. Vaccine sent = $300
    # 2. Vaccine wanted (y_true=1) but NOT sent (send_vaccine=0) = $300 penalty
    # 3. Vaccine not wanted and not sent = $0
    vax_costs = np.where((send_vaccine == 1) | (y_true == 1), COST_VACCINE, 0.0)

    # Hospitalization cost calculation based on actual vaccination status achieved
    actual_vaccinated = np.where((send_vaccine == 1) & (y_true == 1), 1, 0)
    hosp_probs = estimate_hospitalization_risk_vectorized(df_raw, actual_vaccinated)
    hosp_costs = hosp_probs * COST_HOSPITAL

    total_cost = np.sum(vax_costs + hosp_costs)
    return total_cost

print("Cost Function & Risk Estimator Initialized.")


## 4. Model Training, Hyperparameter Tuning & Cross-Validation

We train and compare 5 distinct machine learning models using 5-Fold Stratified Cross-Validation:
1. **HistGradientBoostingClassifier**
2. **RandomForestClassifier**
3. **GradientBoostingClassifier**
4. **LogisticRegression**
5. **KNeighborsClassifier**

In [ ]:
# Train-Validation Split for local performance tuning
X_train_split, X_val_split, y_train_split, y_val_split, df_train_sub, df_val_sub = train_test_split(
    X_processed, y, df_train, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

models = {
    'HistGradientBoosting': HistGradientBoostingClassifier(max_iter=150, max_depth=6, random_state=RANDOM_STATE),
    'RandomForest': RandomForestClassifier(n_estimators=200, max_depth=12, random_state=RANDOM_STATE, n_jobs=-1),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=150, max_depth=5, random_state=RANDOM_STATE),
    'LogisticRegression': LogisticRegression(C=1.0, max_iter=1000, random_state=RANDOM_STATE),
    'KNeighbors': KNeighborsClassifier(n_neighbors=15, weights='distance')
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
model_results = []

for name, model in models.items():
    cv_auc = cross_val_score(model, X_processed, y, cv=cv, scoring='roc_auc')
    model.fit(X_train_split, y_train_split)
    val_probs = model.predict_proba(X_val_split)[:, 1]
    val_auc = roc_auc_score(y_val_split, val_probs)

    best_th = 0.5
    min_cost = float('inf')
    for th in np.linspace(0.1, 0.9, 81):
        cost = compute_town_cost_for_predictions(df_val_sub, y_val_split, val_probs, threshold=th)
        if cost < min_cost:
            min_cost = cost
            best_th = th

    model_results.append({
        'Model': name,
        'CV AUC (Mean)': np.mean(cv_auc),
        'CV AUC (Std)': np.std(cv_auc),
        'Val AUC': val_auc,
        'Optimal Threshold': round(best_th, 2),
        'Min Val Cost ($)': min_cost
    })

df_model_comparison = pd.DataFrame(model_results).sort_values(by='Min Val Cost ($)', ascending=True)
print("=== Model Performance Comparison & Cost Analysis ===")
print(df_model_comparison.to_string(index=False))


## 5. Performance Analysis, ROC Curves & Feature Importance

We plot ROC curves to visualize trade-offs and extract feature importances from our ensemble models.

In [ ]:
# Plot ROC Curves for all 5 models
plt.figure(figsize=(9, 6))
for name, model in models.items():
    val_probs = model.predict_proba(X_val_split)[:, 1]
    fpr, tpr, _ = roc_curve(y_val_split, val_probs)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc_score(y_val_split, val_probs):.4f})")

plt.plot([0, 1], [0, 1], 'k--', label='Random Chance')
plt.title('Receiver Operating Characteristic (ROC) Curves')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


In [ ]:
# Feature Importance Analysis (RandomForest)
ohe_cat_features = preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(cat_cols)
all_feature_names = num_cols + list(ohe_cat_features)

rf_model = models['RandomForest']
importances = rf_model.feature_importances_
df_imp = pd.DataFrame({'Feature': all_feature_names, 'Importance': importances}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=df_imp.head(12), palette='mako')
plt.title('Top 12 Most Important Features for Vaccine Acceptance (Random Forest)')
plt.xlabel('Relative Feature Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


## 6. Test Set Predictions & Top 5 Submission File Generation

We fit each of the top 5 models on the full training dataset (`df_train`) and generate predictions for the hidden test set (`dataset_A_testing.csv`).
We export the top 5 results ranked from Best to Worst in accordance with Canvas submission requirements.

In [ ]:
# Re-fit models on full dataset and generate test predictions
submission_dict = {'respondent_id': df_test['respondent_id']}
ranked_submissions = []

for idx, row in df_model_comparison.reset_index(drop=True).iterrows():
    name = row['Model']
    th = row['Optimal Threshold']
    model = models[name]

    model.fit(X_processed, y)
    test_probs = model.predict_proba(X_test_processed)[:, 1]
    test_preds = (test_probs >= th).astype(int)

    col_prob_name = f"pred_prob_{name}"
    col_class_name = f"pred_class_{name}"

    submission_dict[col_prob_name] = np.round(test_probs, 4)
    submission_dict[col_class_name] = test_preds

    single_sub = pd.DataFrame({
        'respondent_id': df_test['respondent_id'],
        'seasonal_vaccine_prob': np.round(test_probs, 4),
        'seasonal_vaccine_pred': test_preds
    })
    filename = f"submission_rank_{len(ranked_submissions)+1}_{name}.csv"
    single_sub.to_csv(filename, index=False)
    ranked_submissions.append((filename, name, row['Min Val Cost ($)']))
    print(f"Saved Submission Rank {len(ranked_submissions)}: {filename} (Model: {name})")

df_combined_sub = pd.DataFrame(submission_dict)
df_combined_sub.to_csv('top_5_models_predictions_master.csv', index=False)
print("\nMaster submission file saved: 'top_5_models_predictions_master.csv'")

print("\n=== FINAL SUBMISSION RANKING (BEST TO WORST) ===")
for rank, (fname, mname, cost) in enumerate(ranked_submissions, 1):
    print(f"Rank {rank}: {fname} | Model: {mname:20s} | Validation Estimated Cost: ${cost:,.2f}")
